# Lab 18 solution, notebook 1: let a model split the work, and ask what the notes leave out

The worked answers to the two challenge tasks of `18a_meet_agent_designs.ipynb`. The setup repeats the notebook's
blurbs, tool and agents, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import asyncio
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import AsyncOpenAI
from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, Runner, function_tool, set_default_openai_client

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))
SETTINGS = ModelSettings(reasoning=Reasoning(effort=EFFORT))

In [3]:
CLUB = ("Our book club meets in three weeks. What we want from the next book: no more than 350 pages; short "
        "chapters, because most of us read on the bus; a happy ending, because last month's book made half of us "
        "cry; and a book that stands on its own, because we have not read anything else by these authors.")

BOOKS = {}
for path in sorted((ROOT / "18_Agent_Designs" / "book_club").glob("*.md")):
    text = path.read_text(encoding="utf-8")
    BOOKS[text.splitlines()[0].removeprefix("# ")] = text


def read_blurb(title: str) -> str:
    """Read the publisher's blurb of one book on the club's shortlist.

    Args:
        title: The book's title, exactly as it is on the shortlist.
    """
    for known_title, text in BOOKS.items():
        if known_title.lower() == title.strip().lower():
            return text
    return f"There is no book called {title!r} on the shortlist."


READ_BLURB = function_tool(read_blurb)
QUESTION = "Which of these five books should our book club read next? " + "; ".join(BOOKS) + "."

helper = Agent(name="Book club helper", model=MODEL, tools=[READ_BLURB], model_settings=SETTINGS,
               instructions=("You help a book club choose its next book. Read the blurbs with your tool before you "
                             "answer. " + CLUB + " Answer in a few sentences."))
reader = Agent(name="Blurb reader", model=MODEL, tools=[READ_BLURB], model_settings=SETTINGS,
               instructions=("You read the blurb of one book for a book club. " + CLUB + " Read the blurb with your "
                             "tool, then reply with one line: the title, whether it fits, and why."))
chooser = Agent(name="Book club chooser", model=MODEL, model_settings=SETTINGS,
                instructions=("You help a book club choose its next book. " + CLUB + " Answer from the notes you are "
                              "given. Answer in a few sentences."))
chooser_with_tool = Agent(
    name="Chooser with a reader tool",
    instructions=chooser.instructions + " Ask the blurb reader about each book.",
    model=MODEL,
    tools=[reader.as_tool(tool_name="ask_blurb_reader",
                          tool_description="Ask the blurb reader whether one book on the shortlist fits the club.")],
    model_settings=SETTINGS,
)
print([agent.name for agent in [helper, reader, chooser, chooser_with_tool]])

['Book club helper', 'Blurb reader', 'Book club chooser', 'Chooser with a reader tool']


## Challenge 1: let a model split the work

The chooser of Part D calls the reader as a tool. Nothing in our code says how many readers to ask, or for which
books: the chooser's model decides.

In [4]:
led = await Runner.run(chooser_with_tool, QUESTION)

print(led.final_output)
print()
print("requests in the whole run:", led.context_wrapper.usage.requests)
for number, response in enumerate(led.raw_responses, start=1):
    asked = [getattr(part, "name", part.type) for part in response.output]
    print(f"chooser request {number}: {response.usage.input_tokens:4} input tokens -> {asked}")

Read **The Lighthouse Bakery**: it’s 288 pages, has short chapters, ends joyfully, and stands alone. None of the other four meets every requirement.

requests in the whole run: 12
chooser request 1:  215 input tokens -> ['ask_blurb_reader', 'ask_blurb_reader', 'ask_blurb_reader', 'ask_blurb_reader', 'ask_blurb_reader']
chooser request 2:  861 input tokens -> ['message']


The chooser picked *The Lighthouse Bakery*, as both designs of the notebook did. Its first request asked for `ask_blurb_reader` five times in one reply, so the SDK ran the five readers at the same time, and its second request wrote the answer. The whole run took 12 requests: the chooser's 2, and 2 for each of the five readers.

The chooser's answering request read 861 tokens: more than Part C's chooser (320), because it also carries its five tool calls and the input it wrote for each, and less than half of Part B's agent (2,083), because it never sees a blurb. Here the model decided the split; in Part C our code did. Both keep the blurbs away from the agent that answers.

## Challenge 2: ask what the notes leave out

First the five readers and their notes, as in Part C; then the same new question to the chooser, with the notes, and
to the one agent of Part B, which reads the blurbs itself.

In [5]:
reader_runs = await asyncio.gather(
    *[Runner.run(reader, f"Read '{title}' and tell us whether it fits.") for title in BOOKS])
notes = "\n".join(run.final_output for run in reader_runs)
print(notes)

A Short History of Clouds — **doesn’t fit**: it’s 224 pages and ends hopefully, but its six dense, roughly 40-page chapters aren’t suited to short bus rides, and the blurb doesn’t confirm it stands alone.
*Return to Harbour Street* — No; it’s 304 pages, has short chapters, and a happy ending, but it’s the second book in a series and relies on the first.
Salt and Saffron — **Doesn’t fit**: it’s 236 pages with short chapters and stands alone, but its tearful goodbye ending isn’t happy.
The Lighthouse Bakery — Fits: 288 pages, short bus-friendly chapters, a joyful and hopeful ending, and it stands alone.
**The Night Ferry — No, it doesn’t fit:** 412 pages, long chapters, and the blurb doesn’t confirm a happy ending; it does sound like a self-contained story.


In [6]:
RECIPES = "Which of these books has recipes in it?"

from_notes = await Runner.run(chooser, RECIPES + "\n\nNotes from the readers:\n" + notes)
print("chooser:", from_notes.final_output)
print()
from_blurbs = await Runner.run(helper, RECIPES + " " + "; ".join(BOOKS) + ".")
print("helper: ", from_blurbs.final_output)

chooser: The notes don’t say whether any of these books contain recipes.



helper:  *Salt and Saffron* has recipes at the end of each chapter. *The Lighthouse Bakery* also features a recipe notebook in the story, though its blurb doesn’t say the book includes recipes.


No reader wrote about recipes, because the club's wishes say nothing about cooking, so the chooser can only say the notes do not tell. The helper reads the blurbs again and names *Salt and Saffron*, whose recipes come at the end of each chapter (and mentions the recipe notebook in *The Lighthouse Bakery*'s story). Each reader wrote down only what it was asked to look for, and the rest of the blurb did not reach its note. A split works for questions you know before the readers run.